# ImageNet-1K Validation Set — Slipstream Cache Preparation

Build a slipstream cache from raw ImageNet validation images with
**s256_l512** preprocessing (resize short edge to 256, center-crop
long edge to 512, JPEG quality 100).

Two cache formats are built:
1. **JPEG** — standard format, smaller on disk
2. **YUV420** — native YUV planes, faster decode for some decoders

## Prerequisites
- Raw ImageNet val at `IMAGENET_ROOT/val/` with 1000 class subdirectories
- slipstream on the `feature/parallel-cache-build` branch

## Configuration

In [ ]:
import os
from pathlib import Path

IMAGENET_ROOT = os.environ.get(
    "IMAGENET_ROOT",
    "/Users/gaa019/Datasets/imagenet1k/rawdata",
)
SPLIT = "val"
NUM_WORKERS = max(os.cpu_count() - 1, 1)

# Output directories (set to None to use default cache_path)
JPEG_OUTPUT_DIR = Path(os.environ.get(
    "JPEG_OUTPUT_DIR",
    os.path.expanduser("~/.slipstream/imagenet1k-s256_l512-jpeg-val"),
))
YUV_OUTPUT_DIR = Path(os.environ.get(
    "YUV_OUTPUT_DIR",
    os.path.expanduser("~/.slipstream/imagenet1k-s256_l512-yuv420-val"),
))

print(f"ImageNet root: {IMAGENET_ROOT}")
print(f"Split: {SPLIT}")
print(f"Workers: {NUM_WORKERS}")
print(f"JPEG output: {JPEG_OUTPUT_DIR}")
print(f"YUV output: {YUV_OUTPUT_DIR}")

## Step 1: Create the prep dataset

In [ ]:
from slipstream.prep import ImageNet1k_s256_l512

dataset = ImageNet1k_s256_l512(IMAGENET_ROOT, split=SPLIT)
print(dataset)

### Quick sanity check — inspect a single sample

In [ ]:
sample = dataset[0]
print(f"Fields: {list(sample.keys())}")
print(f"Image: {len(sample['image']):,} bytes (JPEG)")
print(f"Label: {sample['label']}")
print(f"Index: {sample['index']}")
print(f"Path: {sample['path']}")

# Verify it's valid JPEG
assert sample['image'][:2] == b'\xff\xd8', "Not a JPEG!"
print("\nJPEG header OK ✓")

In [ ]:
# Visualize the preprocessed image
from PIL import Image
import io

img = Image.open(io.BytesIO(sample['image']))
print(f"Dimensions: {img.size[0]}x{img.size[1]}")
img

In [ ]:
from slipstream.cache import OptimizedCache

try:
    jpeg_cache = OptimizedCache.load(JPEG_OUTPUT_DIR)
    yuv_cache = OptimizedCache.load(YUV_OUTPUT_DIR)
    print(jpeg_cache, yuv_cache)
except:
    print("not yet available...")

## Step 2: Build JPEG cache (parallel)

In [ ]:
import time
from slipstream.cache import OptimizedCache

print(f"Building JPEG cache with {NUM_WORKERS} workers...")
t0 = time.time()
jpeg_cache = OptimizedCache.build(
    dataset,
    output_dir=JPEG_OUTPUT_DIR,
    num_workers=NUM_WORKERS,
    image_format="jpeg",
)
elapsed = time.time() - t0
print(f"\nDone: {elapsed:.1f}s ({len(dataset)/elapsed:.0f} samples/sec)")

In [ ]:
# Cache summary
print(f"Samples: {jpeg_cache.num_samples:,}")
print(f"Fields: {list(jpeg_cache.field_types.keys())}")
print(f"\nFile sizes:")
for f in sorted(jpeg_cache.cache_dir.iterdir()):
    if f.is_file():
        print(f"  {f.name:30s} {f.stat().st_size/1e6:8.1f} MB")

## Step 3: Build YUV420 cache (parallel)

In [ ]:
print(f"Building YUV420 cache with {NUM_WORKERS} workers...")
t0 = time.time()
yuv_cache = OptimizedCache.build(
    dataset,
    output_dir=YUV_OUTPUT_DIR,
    num_workers=NUM_WORKERS,
    image_format="yuv420",
)
elapsed = time.time() - t0
print(f"\nDone: {elapsed:.1f}s ({len(dataset)/elapsed:.0f} samples/sec)")

In [ ]:
# Cache summary
print(f"Samples: {yuv_cache.num_samples:,}")
print(f"Fields: {list(yuv_cache.field_types.keys())}")
print(f"\nFile sizes:")
for f in sorted(yuv_cache.cache_dir.iterdir()):
    if f.is_file():
        print(f"  {f.name:30s} {f.stat().st_size/1e6:8.1f} MB")

## Step 4: Verify both caches

In [ ]:
import numpy as np

indices = np.array([0, 100, 1000, 10000, 49999], dtype=np.int64)

jpeg_batch = jpeg_cache.load_batch(indices, fields=['image', 'label'])
yuv_batch = yuv_cache.load_batch(indices, fields=['image', 'label'])

print("JPEG cache spot-check:")
for i, idx in enumerate(indices):
    size = jpeg_batch['image']['sizes'][i]
    label = jpeg_batch['label']['data'][i]
    print(f"  [{idx:5d}] image={size:6d} bytes, label={label}")

print("\nYUV420 cache spot-check:")
for i, idx in enumerate(indices):
    size = yuv_batch['image']['sizes'][i]
    label = yuv_batch['label']['data'][i]
    print(f"  [{idx:5d}] image={size:6d} bytes, label={label}")

# Labels should match between formats
np.testing.assert_array_equal(
    jpeg_batch['label']['data'],
    yuv_batch['label']['data'],
)
print("\nLabels match across formats ✓")

## Step 5: Build indexes

In [ ]:
from slipstream.cache import write_index

# Build label index for both caches
for name, cache in [("JPEG", jpeg_cache), ("YUV420", yuv_cache)]:
    write_index(cache, fields=["label"])
    print(f"{name}: label index built")

In [ ]:
jpeg_cache.cache_dir, yuv_cache.cache_dir

## Step 6: (Optional) Sync to remote S3 cache

Uncomment to upload the caches to S3 for lab-wide sharing.

In [ ]:
from slipstream.s3_sync import upload_s3_cache

REMOTE_CACHE = "s3://visionlab-datasets/slipstream-cache/imagenet1k/imagenet1k-s256_l512-jpeg-val/"

upload_s3_cache(
    jpeg_cache,
    REMOTE_CACHE,
)
print(f"JPEG cache uploaded to {REMOTE_CACHE}")

In [ ]:
REMOTE_CACHE = "s3://visionlab-datasets/slipstream-cache/imagenet1k/imagenet1k-s256_l512-yuv420-val/"

upload_s3_cache(
    yuv_cache,
    REMOTE_CACHE,
)
print(f"YUV cache uploaded to {REMOTE_CACHE}")

## Summary

| Format | Samples | Size | Build Time |
|--------|---------|------|------------|
| JPEG   | 50,000  | ~3.8 GB | ~34s (13 workers) |
| YUV420 | 50,000  | ~4.8 GB | ~40s (13 workers) |

Both caches are ready for use with `SlipstreamLoader`:

```python
from slipstream import SlipstreamDataset, SlipstreamLoader
from slipstream.pipelines import supervised_val

dataset = SlipstreamDataset(local_dir=str(JPEG_OUTPUT_DIR))
loader = SlipstreamLoader(dataset, batch_size=256, pipelines=supervised_val(224))
```

## Step 7: Load and view samples

In [ ]:
from slipstream import SlipstreamDataset, SlipstreamLoader
from slipstream.pipelines import supervised_val

jpeg_dataset = SlipstreamDataset(local_dir=str(JPEG_OUTPUT_DIR), decode_images=True)
jpeg_dataset

In [ ]:
sample = jpeg_dataset[0]
sample.keys()

In [ ]:
print(sample['label'], sample['path'])
sample['image']

In [ ]:
yuv_dataset = SlipstreamDataset(local_dir=str(YUV_OUTPUT_DIR), decode_images=True)
yuv_dataset

In [ ]:
sample = yuv_dataset[0]
sample.keys()

In [ ]:
print(sample['label'], sample['path'])
sample['image']

In [ ]:
supervised_val(224)

In [ ]:
# Load a batch from the JPEG cache
pipelines = supervised_val(224)
jpeg_loader = SlipstreamLoader(
    jpeg_dataset,
    batch_size=16,
    pipelines=pipelines,
)

batch = next(iter(jpeg_loader))
print(f"Images: {batch['image'].shape}, dtype={batch['image'].dtype}")
print(f"Labels: {batch['label'].shape}")

In [ ]:
# Visualize the batch
from slipstream import show_batch

mean = pipelines['image'][-1].mean.numpy()
std = pipelines['image'][-1].std.numpy()
show_batch(batch['image'], batch['label'], n_cols=8, mean=mean, std=std)

In [ ]:
# Load a batch from the YUV420 cache
pipelines = supervised_val(224)
mean = pipelines['image'][-1].mean.numpy()
std = pipelines['image'][-1].std.numpy()

yuv_loader = SlipstreamLoader(
    yuv_dataset,
    batch_size=16,
    pipelines=pipelines,
)

batch_yuv = next(iter(yuv_loader))
print(f"Images: {batch_yuv['image'].shape}, dtype={batch_yuv['image'].dtype}")

show_batch(batch_yuv['image'], batch_yuv['label'], n_cols=8, mean=mean, std=std)

In [ ]:
# show_batch?